# AI Surrogate Model for Automotive Hood Deformation

## From CAD parameters and 3D geometry to a multimodal CAE surrogate

### Objective
Can machine learning predict maximum hood deformation for a new design without running the full FEA analysis?

The common dataset exploration and PCA are documented separately in:

**`00_CarHoods10k_PCA_EDA.ipynb`**

The condensed dataset provides:
- **54 CAD/topological parameters**
- **8,192 XYZ points per hood geometry**
- deformation, stress and mass FEA targets

The HDF5 file does not retain the exact mapping between P1–P54 and the original CAD parameter names, so unsupported physical names are not assigned.

### Selected deformation model
The final surrogate combines:
- PointNet-style 3D geometry features,
- all 54 CAD parameters,
- engineered geometry descriptors.

Held-out random-design result:

- **R² = 0.9739**
- **MAE = 0.3467 mm**
- **RMSE = 0.6896 mm**

> This benchmark represents held-out designs from the available dataset. It is not a topology-disjoint deployment test.

## Engineering context

In conventional CAE development, evaluating a new design requires geometry preparation, meshing, solver execution, post-processing, and engineering review. When thousands of design variants must be evaluated, repeated CAE runs become expensive.

A surrogate model can reduce turnaround time by learning the relationship:

**Design + Geometry → Structural response**

For deformation, the central question is:

> Can deformation be predicted accurately enough from design parameters and geometric information to support rapid design screening?

In [ ]:
# Core imports
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Load the CAE dataset

The H5 dataset contains:
- `design_parameters`: 54 parametric design variables
- `points`: 3D point cloud for each hood geometry
- `deformation`: scalar deformation response
- `stress`: scalar stress response
- `mass`: scalar hood mass

The study uses the same data source throughout so that model comparisons remain consistent.

In [ ]:
FILE_PATH = "/content/drive/MyDrive/AI-CAE-Project/CarHoods_Extracted.h5"

with h5py.File(FILE_PATH, "r") as f:
    print("H5 keys:", list(f.keys()))
    design_parameters = f["design_parameters"][:]
    points = f["points"][:]
    deformation = f["deformation"][:]
    stress = f["stress"][:]
    mass = f["mass"][:]

print("design_parameters:", design_parameters.shape)
print("points:", points.shape)
print("deformation:", deformation.shape)
print("stress:", stress.shape)
print("mass:", mass.shape)

### Expected dataset dimensions

The original dataset contains approximately:

- **9,982 designs**
- **54 design parameters**
- **8,192 3D points per hood**

A small number of point clouds are invalid because all coordinates are zero. These are removed before geometry-based modeling.

In [ ]:
# Detect invalid all-zero point clouds
valid_mask = np.any(np.abs(points) > 0, axis=(1, 2))

print("Valid designs:", valid_mask.sum())
print("Invalid designs:", (~valid_mask).sum())

X_params = design_parameters[valid_mask]
X_points = points[valid_mask]
y_def = deformation[valid_mask]
y_stress = stress[valid_mask]
y_mass = mass[valid_mask]

print("Clean shapes:")
print("Params:", X_params.shape)
print("Points:", X_points.shape)
print("Deformation:", y_def.shape)

## 2. Exploratory Data Analysis

The EDA has three purposes:

1. Understand the response distribution.
2. Understand the nature of the 54 design variables.
3. Determine whether geometry contains information that is not captured by the design parameters.

This is important because a surrogate model should be guided by the physics and structure of the data, not only by algorithm selection.

In [ ]:
# Deformation summary statistics
pd.Series(y_def, name="Deformation_mm").describe()

In [ ]:
plt.figure(figsize=(7,4))
plt.hist(y_def, bins=40)
plt.xlabel("Deformation [mm]")
plt.ylabel("Count")
plt.title("Distribution of Hood Deformation")
plt.show()

### Deformation distribution

Typical deformation statistics observed in the dataset are approximately:

- Mean: **10.10 mm**
- Standard deviation: **4.19 mm**
- Minimum: **1.96 mm**
- Median: **9.82 mm**
- Maximum: **27.29 mm**

The response is continuous and broad enough to support regression modeling.

In [ ]:
# Parameter statistics
param_cols = [f"P{i+1}" for i in range(X_params.shape[1])]
df_params = pd.DataFrame(X_params, columns=param_cols)

summary = df_params.describe().T
summary[["mean", "std", "min", "25%", "50%", "75%", "max"]].head(20)

## 3. Sparse and active design variables

Many design parameters have medians at zero. This means several variables behave more like **design knobs that are activated only for selected variants** rather than continuously varying dimensions.

This is a useful engineering observation because:
- zero/non-zero state may indicate a design configuration,
- parameter activation patterns can represent design families,
- identical parameter magnitudes may have different meaning in different geometries.

In [ ]:
zero_fraction = (df_params == 0).mean().sort_values(ascending=False)

plt.figure(figsize=(12,4))
zero_fraction.plot(kind="bar")
plt.ylabel("Fraction equal to zero")
plt.title("Sparsity of the 54 Design Parameters")
plt.tight_layout()
plt.show()

## EDA and PCA findings carried into the deformation study

The separate EDA notebook showed that the 54-dimensional CAD space contains structured parameter activity and dominant latent directions.

PCA is intentionally **not used to reduce the final model inputs**. It is used to understand the design space.

Why retain all 54 parameters?

- low-variance directions can still matter to structural response,
- PCA components mix several original CAD variables and are less directly interpretable,
- parameter activation can contain topology/configuration information,
- most importantly, PCA of the parameters cannot replace the actual 3D geometry.

This leads to the central modeling hypothesis:

> **CAD parameters and resulting physical geometry contain complementary information.**

## 4. Parameter-only baseline

The first benchmark intentionally ignores the 3D geometry.

This answers:

> How much deformation information is already encoded in the 54 design parameters?

A simple linear model is useful here because it establishes a transparent baseline before adding geometric complexity.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_params, y_def, test_size=0.20, random_state=SEED
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_s, y_train)
pred = lr.predict(X_test_s)

print("Parameter-only Linear Regression")
print("MAE :", mean_absolute_error(y_test, pred))
print("RMSE:", mean_squared_error(y_test, pred) ** 0.5)
print("R2  :", r2_score(y_test, pred))

### Baseline result

In the earlier benchmark, the parameter-only linear model achieved approximately:

**R² ≈ 0.286**

This is an important result.

The 54 design variables do contain information, but they do **not** fully describe structural deformation.

This suggested that hidden geometric variation was important.

## 5. Engineering geometry descriptors

Instead of immediately moving to a deep neural network, simple geometric descriptors were extracted from the full point cloud.

Typical descriptors include:
- X, Y, Z span
- bounding-box dimensions
- centroid-related measures
- radial extent
- point-cloud standard deviations
- overall size-related quantities

These descriptors provide a low-dimensional description of the physical geometry.

In [ ]:
def geometry_descriptors(point_clouds):
    feats = []
    for pts in point_clouds:
        xyz_min = pts.min(axis=0)
        xyz_max = pts.max(axis=0)
        spans = xyz_max - xyz_min
        centroid = pts.mean(axis=0)
        stds = pts.std(axis=0)
        radial = np.linalg.norm(pts - centroid, axis=1)

        feats.append([
            spans[0], spans[1], spans[2],
            stds[0], stds[1], stds[2],
            radial.mean(),
            radial.std(),
            radial.max()
        ])

    return np.asarray(feats, dtype=np.float32)

X_geo = geometry_descriptors(X_points)
print("Geometry descriptor shape:", X_geo.shape)

### Why geometry descriptors matter

The geometry descriptors substantially improved prediction.

Earlier experiments showed approximately:

| Inputs | R² |
|---|---:|
| 54 design parameters | 0.286 |
| geometry descriptors | 0.435 |
| parameters + geometry | **0.721** |

This was the first strong indication that **geometry is not merely supplementary information — it is a central part of the deformation mapping**.

In [ ]:
X_combined = np.hstack([X_params, X_geo])

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y_def, test_size=0.20, random_state=SEED
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_s, y_train)
pred = lr.predict(X_test_s)

print("Parameters + geometry descriptors")
print("MAE :", mean_absolute_error(y_test, pred))
print("RMSE:", mean_squared_error(y_test, pred) ** 0.5)
print("R2  :", r2_score(y_test, pred))

## 6. Pattern-disjoint validation

A random row split can place similar design configurations in both training and test data.

To make the test more demanding, parameter activation patterns can be used to create a **pattern-disjoint split**, where entire activation patterns are separated.

This is not a full unseen-topology validation, but it is a stronger test of interpolation across design configurations.

In [ ]:
# Activation pattern example
activation = (X_params != 0).astype(np.int8)
pattern_strings = np.array(["".join(row.astype(str)) for row in activation])

pattern_counts = pd.Series(pattern_strings).value_counts()
print("Unique activation patterns:", len(pattern_counts))
pattern_counts.head()

Earlier pattern-disjoint experiments produced approximately:

| Inputs | Pattern-disjoint R² |
|---|---:|
| parameters | 0.196 |
| geometry descriptors | 0.504 |
| parameters + geometry | **0.808** |

The relative ranking remained the same:

> Geometry carries essential structural information that the parameter vector alone cannot recover.

## 7. Geometry-family investigation

Clustering the geometric descriptors provided another useful insight.

When a geometry cluster label was supplied together with the parameters, deformation prediction improved further.

A representative experiment achieved:

**R² ≈ 0.893**

when using the actual geometry cluster together with the design variables.

However, when the cluster itself had to be inferred from the parameters alone, performance dropped substantially.

### Engineering interpretation

The design parameter vector does not uniquely determine the base geometry family.

This is why geometry must be represented explicitly rather than treated as something the network can always reconstruct from the 54 variables.

## 8. Move to 3D point-cloud learning

The next step was to allow the model to learn directly from the 3D hood geometry instead of compressing it into only a few descriptors.

A PointNet-style encoder was selected because:
- point clouds are unordered,
- mesh connectivity was not required,
- it is computationally simpler than full graph-based processing,
- max pooling provides permutation-invariant global geometry features.

### Why the selected deformation model uses standard PyTorch rather than PyTorch Geometric

The selected encoder is PointNet-style.

Each XYZ point passes through the same shared `Conv1d(kernel_size=1)` transformations. Global max and mean pooling then create a permutation-invariant geometry representation.

No mesh edges, radius graph or local neighborhood construction is required. Therefore the final deformation model can be implemented entirely in standard **PyTorch**.

`torch_geometric` and `torch_cluster` were introduced only in the later PointNet++ challenge experiments, where farthest-point sampling, radius searches and local `PointNetConv` operations were required.

In [ ]:
# Fixed point sampling for reproducibility
rng = np.random.default_rng(SEED)

N_POINTS = 1024
sample_idx = rng.choice(X_points.shape[1], size=N_POINTS, replace=False)

points_1024 = X_points[:, sample_idx, :].astype(np.float32)
print(points_1024.shape)

## 9. An important engineering decision: preserve physical scale

An early experiment normalized every hood independently to a unit radius.

That appeared mathematically convenient, but it removed absolute physical dimensions.

For a structural problem, this is dangerous because:
- panel span affects stiffness,
- structural dimensions affect deformation,
- two geometrically similar but differently sized hoods should not become identical after normalization.

### Engineering judgement

Instead of unit-sphere normalization, each hood was:

1. centered at its own centroid,
2. kept at its original physical size,
3. scaled globally by `/1000` only to put the coordinates into numerically convenient units.

This preserved dimensional information while stabilizing neural-network training.

In [ ]:
def center_and_scale(points_array):
    centered = points_array - points_array.mean(axis=1, keepdims=True)
    return centered / 1000.0

points_1024_scaled = center_and_scale(points_1024)

### PointNet benchmark

With centered point clouds and physical scale preserved, PointNet achieved approximately:

- **MAE = 0.924 mm**
- **RMSE = 1.183 mm**
- **R² = 0.923**

on a random row split.

On a more difficult geometry-aware / pattern split:

- **MAE ≈ 0.72 mm**
- **RMSE ≈ 0.94 mm**
- **R² ≈ 0.851**

This confirmed that raw 3D geometry contains strong predictive information.

## 10. Point-count experiment

More points did not automatically improve performance.

Representative deformation experiments showed:

| Sampled points | R² |
|---:|---:|
| 1,024 | **0.859** |
| 2,048 | 0.766 |
| 4,096 | 0.774 |

This is useful because it shows that model quality is not simply controlled by point density.

For this dataset, 1,024 points provided a better balance between:
- geometry information,
- optimization stability,
- computational cost.

## 11. Final multimodal deformation model

The final model combines three complementary information sources:

### 3D point-cloud branch
Learns the overall geometric shape directly.

### Design-parameter branch
Learns the effect of the 54 explicit engineering parameters.

### Geometry-descriptor branch
Supplies physically meaningful low-dimensional size and shape indicators.

The three feature vectors are fused before the final regression head.

In [ ]:
class PointEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(3, 64, 1),
            nn.ReLU(),
            nn.Conv1d(64, 128, 1),
            nn.ReLU(),
            nn.Conv1d(128, 256, 1),
            nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU()
        )

    def forward(self, x):
        # x: [B, N, 3]
        x = x.transpose(1, 2)
        f = self.net(x)
        f_max = torch.max(f, dim=2).values
        f_mean = torch.mean(f, dim=2)
        f = torch.cat([f_max, f_mean], dim=1)
        return self.fc(f)


class ParamEncoder(nn.Module):
    def __init__(self, n_params=54):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_params, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


class GeoEncoder(nn.Module):
    def __init__(self, n_geo=9):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_geo, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


class DeformationFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.point_encoder = PointEncoder()
        self.param_encoder = ParamEncoder()
        self.geo_encoder = GeoEncoder()

        self.fusion = nn.Sequential(
            nn.Linear(128 + 128 + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, points, params, geo):
        fp = self.point_encoder(points)
        fd = self.param_encoder(params)
        fg = self.geo_encoder(geo)
        return self.fusion(torch.cat([fp, fd, fg], dim=1)).squeeze(1)

## 12. Final result

The final multimodal model achieved:

- **MAE = 0.3467 mm**
- **RMSE = 0.6896 mm**
- **R² = 0.9739**

This represents a large improvement over the parameter-only baseline.

### Progression

| Model | R² |
|---|---:|
| Parameters only, linear | 0.286 |
| Geometry descriptors only | 0.435 |
| Parameters + descriptors | 0.721 |
| PointNet | 0.923 |
| Final multimodal model | **0.974** |

In [ ]:
# Final benchmark values from the validated experiment
results = pd.DataFrame({
    "Model": [
        "Parameter-only Linear Regression",
        "Geometry descriptors",
        "Parameters + geometry descriptors",
        "PointNet",
        "Final multimodal model"
    ],
    "R2": [0.2855, 0.4345, 0.7213, 0.9232, 0.9739]
})

results

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(results["Model"], results["R2"])
plt.ylabel("R²")
plt.ylim(0, 1.0)
plt.xticks(rotation=35, ha="right")
plt.title("Deformation Model Progression")
plt.tight_layout()
plt.show()

## 13. Engineering conclusions

This study produced several practical findings.

### 1. Geometry is essential
The 54 design parameters alone did not contain enough information to explain deformation.

### 2. Physical scaling matters
Per-design unit-sphere normalization removed structural size information and degraded robustness. Preserving physical scale was a meaningful engineering decision.

### 3. More point-cloud resolution is not always better
Increasing the number of sampled points did not consistently improve performance.

### 4. Multimodal fusion was more effective than relying on one representation
The best performance came from combining:
- learned 3D geometry features,
- explicit design variables,
- engineered geometry descriptors.

### 5. Validation scope must be stated correctly
The final R² of 0.974 represents held-out designs from the available dataset. It should not be presented as proof of generalization to completely unseen vehicle programs or topology families.

## 14. Project takeaway

The deformation study demonstrates that a CAE surrogate can achieve high predictive accuracy when the model representation respects the physics of the engineering problem.

The main improvement did not come from increasing network complexity alone.

It came from recognizing that:

> **The design vector and the geometry describe different parts of the engineering problem, and both must be represented explicitly.**

## Documentation summary

The complete deformation workflow is now documented in two layers:

**`00_CarHoods10k_PCA_EDA.ipynb`**  
Explains the dataset, parameter activity, response distributions, PCA, PCA loadings and design-space structure.

**`01_Deformation_Surrogate_REVISED.ipynb`**  
Uses those findings to explain the modeling progression from parameter-only baselines to geometry-aware and point-cloud models and finally to the multimodal surrogate.

This separation keeps EDA conclusions distinct from supervised-model performance and makes the engineering reasoning easier to follow.